# Unsupervised matching

Fixed blocking; embedding-only panels. Select within each panel by record-disjoint validation NLL, then refit on all candidates. After loading the fixed, previously gold-selected candidates, gold is used only for reporting. NLL is not comparable across panels.

Pure transitivity here means descending-score greedy one-to-one without a score cut. Record cosine is an additional block; its conditional independence from attribute similarities is an assumption.

Distribution fitting always uses `transitivity=False`. Thresholds and one-to-one constraints act only on frozen scores. Earlier in-model-transitivity ablations are excluded from accepted results.
KaLM is the fixed blocker: the prior experiments ranked minimum candidate cost at PC ≥ .90, and KaLM won the remaining mean-PQ tie. FZ/DA were tie-heavy; AB/AG/DS determined the ranks. All 20 embedding models are compared as matching features; the blocker-selection evidence and five primary operating points are recorded at the end. The 19 existing ladder cells are sensitivity checks.


In [1]:
import os
for name in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[name] = "1"
from pathlib import Path
import contextlib
import json
import warnings
import optuna
import pandas as pd
from joblib import Parallel, delayed
from config import LADDER, OUTPUT_DIR, UNSUPERVISED_STORAGE
from data import FAMILIES
from unsupervised import run_search, run_refit, _storage
optuna.logging.set_verbosity(optuna.logging.WARNING)

STORAGE = UNSUPERVISED_STORAGE
EXPERIMENT = "unsupervised-v1"
FINAL_EXPERIMENT = "unsupervised-refined-v1"
BASELINE_EXPERIMENT = "unsupervised-baselines-v2"
SPLIT = dict(n_splits=3, seed=0)
THRESHOLDS = dict(llr=[-5., -2., -1., 0., 1., 2., 5.],
                  posterior=[.1, .25, .5, .75, .9], cosine=[.3, .5, .7, .8, .9])
SCORES = Path(OUTPUT_DIR) / "unsupervised_scores"
LOGS = Path(OUTPUT_DIR) / "unsupervised_logs"
MODELS = [model for models in FAMILIES.values() for model in models]
CONFIGS = {
    f"bay={bay}:init={init}": dict(c_bay=bay, init_statistic=init, init_threshold=.5,
        missing="ignore", hard=False, transitivity=False, pi_M=None, tol=1e-5, max_iter=200)
    for bay in (.001, .01, .1) for init in ("sum", "present_mean")
}
PANELS = [dict(name=f"single:{model}", models=[model], fusion="separate", metrics=["cos"])
          for model in MODELS]
PANELS += [dict(name=f"all:{fusion}", models=MODELS, fusion=fusion, metrics=["cos"])
           for fusion in ("separate", "mean", "zmean", "family")]
PANELS += [dict(name="classic:pair", models=["minilm6", "mpnet"], fusion="separate", metrics=["cos"]),
           dict(name="retriever:pair", models=["e5_large_v2", "gte_large"], fusion="separate", metrics=["cos"]),
           dict(name="classic:cos+linf", models=["minilm6", "mpnet"], fusion="separate", metrics=["cos", "linf"])]
PRIMARY_CELLS = [("fodors_zagats", "right", 1), ("dblp_acm", "right", 1),
                 ("abt_buy", "left", 2), ("amazon_googleproducts", "left", 4),
                 ("dblp_scholar", "left", 6)]
SENSITIVITY_CELLS = [(dataset, side, k) for dataset, cells in LADDER.items() for side, k in cells]
CELLS = list(dict.fromkeys(PRIMARY_CELLS + SENSITIVITY_CELLS))
len(CELLS), len(PANELS), len(CONFIGS)

/home/mda4172/Projects/data-env/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


(22, 27, 6)

In [2]:
def run_cell(cell, panel, experiment=EXPERIMENT, ablations=True):
    dataset, side, k = cell
    LOGS.mkdir(parents=True, exist_ok=True)
    label = f"{experiment}__{dataset}__{side}_{k}__{panel['name']}".replace("/", "-")
    with (LOGS / (label + ".log")).open("a") as log, contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
        warnings.simplefilter("once")
        study = run_search(dataset, side, k, panel, CONFIGS, SPLIT, STORAGE, experiment=experiment)
        complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        result = dict(study=study.study_name, complete=len(complete), failed=len(study.trials)-len(complete))
        if complete:
            full = run_refit(study, STORAGE, THRESHOLDS, SCORES)
            result["refit"] = full.trials[-1].state.name
            if ablations and panel["name"] == "single:minilm6":
                for selection, override in [("zero_fill", dict(missing="zero")),
                                             ("hard_em", dict(hard=True))]:
                    arm = run_refit(study, STORAGE, THRESHOLDS, SCORES,
                                    config_override=override, selection=selection)
                    result[selection] = arm.trials[-1].state.name
        return result


def run_matrix(cells=CELLS, panels=PANELS, experiment=EXPERIMENT, workers=4, ablations=True):
    _storage(STORAGE)  # Initialise SQLite before workers start.
    return Parallel(n_jobs=workers, verbose=10)(
        delayed(run_cell)(cell, panel, experiment, ablations) for cell in cells for panel in panels)

The pilot uses DBLP-ACM right/k5 and Abt-Buy left/k2. The 22-cell union has 594 panel/cell studies: 135 at the five primary points and 513 across the 19 sensitivity cells, with two cells shared. Each initial search evaluates six configurations on three folds; refinement selects among ten configurations. Missingness and hard-EM ablations reuse MiniLM6 selections.

Cells with no eligible configuration remain failed. Fixed thresholds and no-cut greedy results are label-free decisions; exact gold-best cuts are oracle diagnostics, not deployable selections.

In [3]:
RUN_PILOT = False
RUN_ALL = False
if RUN_PILOT:
    pilot = run_matrix(cells=[("dblp_acm", "right", 5), ("abt_buy", "left", 2)],
                       panels=[PANELS[0], PANELS[-3]], experiment="unsupervised-pilot-v1", ablations=False)
    display(pd.DataFrame(pilot))
if RUN_ALL:
    completed = run_matrix()
    display(pd.DataFrame(completed))

The initial NLL winners reached the lower regularization boundary. Add `c_bay=0.0001` and `0` with both initializers; reuse the original six trials and select among all ten. This refinement uses NLL only.

In [4]:
from unsupervised_refinement import run_refinement
import sqlite3
from unsupervised import _database

EXTRA_CONFIGS = {
    f"bay={bay}:init={init}": dict(c_bay=bay, init_statistic=init, init_threshold=.5,
        missing="ignore", hard=False, transitivity=False, pi_M=None, tol=1e-5, max_iter=200)
    for bay in (0., .0001) for init in ("sum", "present_mean")
}


def refine_cell(source):
    with (LOGS / ("refinement__" + source + ".log")).open("a") as log, contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
        study = run_refinement(source, STORAGE, EXTRA_CONFIGS, experiment=FINAL_EXPERIMENT)
        complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        result = dict(study=study.study_name, complete=len(complete), failed=len(study.trials)-len(complete))
        if complete:
            full = run_refit(study, STORAGE, THRESHOLDS, SCORES)
            result["refit"] = full.trials[-1].state.name
            if study.user_attrs["panel_name"] == "single:minilm6":
                for selection, override in [("zero_fill", dict(missing="zero")),
                                             ("hard_em", dict(hard=True))]:
                    arm = run_refit(study, STORAGE, THRESHOLDS, SCORES,
                                    config_override=override, selection=selection)
                    result[selection] = arm.trials[-1].state.name
        return result


def run_refinement_matrix(workers=4):
    with sqlite3.connect(_database(STORAGE)) as connection:
        names = [row[0] for row in connection.execute(
            "SELECT s.study_name FROM studies s "
            "JOIN study_user_attributes e ON e.study_id=s.study_id AND e.key='experiment' "
            "JOIN study_user_attributes p ON p.study_id=s.study_id AND p.key='phase' "
            "WHERE json_extract(e.value_json, '$')=? AND json_extract(p.value_json, '$')='search' "
            "ORDER BY s.study_id", (EXPERIMENT,))]
    if len(names) != len(CELLS) * len(PANELS):
        raise ValueError("Complete the initial grid before refinement")
    return Parallel(n_jobs=workers, verbose=10)(delayed(refine_cell)(name) for name in names)


RUN_REFINEMENT = False
if RUN_REFINEMENT:
    refined = run_refinement_matrix()
    display(pd.DataFrame(refined))

Original ZeroER uses the stored string-similarity features on the same candidates. Timings recompute similarities from existing embeddings and exclude embedding generation and blocking.

In [5]:
RUN_BASELINES = False
RUN_TIMINGS = False
if RUN_BASELINES:
    from unsupervised_baseline import run_baselines
    baseline_results = [run_baselines(dataset, side, k, STORAGE, SCORES, thresholds=THRESHOLDS["posterior"],
                                      experiment=BASELINE_EXPERIMENT)
                        for dataset, side, k in CELLS]
if RUN_TIMINGS:
    from unsupervised_timing import benchmark_study
    selected = [s.study_name for s in optuna.get_all_study_summaries(storage=_storage(STORAGE))
                if s.user_attrs.get("experiment") == FINAL_EXPERIMENT
                and s.user_attrs.get("phase") == "full"
                and s.user_attrs.get("role") == "selected"
                and s.best_trial is not None]
    timing_results = [benchmark_study(name, STORAGE, repeats=3) for name in selected]

RUN_COSINE_TIMINGS = False
RUN_STRING_TIMING_SMOKE = False
if RUN_COSINE_TIMINGS:
    from unsupervised_cosine_timing import benchmark_cosine_cell
    cosine_timings = [benchmark_cosine_cell(dataset, side, k, MODELS, STORAGE,
                                            THRESHOLDS["cosine"], repeats=3)
                      for dataset, side, k in CELLS]
if RUN_STRING_TIMING_SMOKE:
    from unsupervised_baseline_timing import benchmark_baseline_cell
    string_timing = benchmark_baseline_cell("fodors_zagats", "right", 1, STORAGE)

## Fixed blocker and primary operating points

Candidates use **KaLM only** (`config.MODEL`). The 20 models vary the matching features on those same candidates. The previous blocker search is closed: for each model/dataset, minimize candidate count subject to **PC ≥ .90**, then prefer higher PQ; direction and k finish ties. Rank models by mean candidate-count rank, worst rank, then mean PQ. KaLM ties F2LLM-14B/8B on mean rank (1.6667) and worst rank (2), and wins the PQ tie-break (.4608 vs .4595/.4584).

The aggregate rank uses Abt–Buy, Amazon–Google and DBLP–Scholar. DBLP–ACM (20/20 first-place ties) and Fodors–Zagats (18/20) are excluded from that rank because more than half the models tie first. Mean PQ still averages all five datasets, as in the original notebook. These blocking choices used gold; matcher selection uses only within-panel CV NLL.

The **five primary points** below are the prior minimum-cost KaLM points at strict PC ≥ .90. The existing **19 ladder cells remain sensitivity checks**. Three primary points are outside that ladder (DA right/k1, AG left/k4, DS left/k6); missing matching results stay explicit. DS left/k5 has PC **.899943893772**, so rounding it to .90 does not qualify it. SQL reads `outputs/optuna.db` without changing it.


In [6]:
import sqlite3
from config import MODEL as BLOCKER_MODEL

BLOCKING_DATABASE = Path(OUTPUT_DIR) / "optuna.db"
with sqlite3.connect(BLOCKING_DATABASE.resolve().as_uri() + "?mode=ro", uri=True) as blocking_evidence_db:
    blocking_evidence_db.executescript("""
    CREATE TEMP VIEW blocking_evidence AS
    WITH params AS (
        SELECT trial_id, json_group_object(param_name,
            CASE WHEN json_extract(distribution_json, '$.name') = 'CategoricalDistribution'
                 THEN json_extract(distribution_json,
                      '$.attributes.choices[' || CAST(param_value AS INTEGER) || ']')
                 ELSE param_value END) AS p
        FROM trial_params GROUP BY trial_id
    ), attrs AS (
        SELECT trial_id, json_group_object(key, json(value_json)) AS a
        FROM trial_user_attributes GROUP BY trial_id
    ), named AS (
        SELECT study_id, CASE WHEN study_name GLOB '*_full'
            THEN substr(study_name, 1, length(study_name)-5) ELSE study_name END AS name
        FROM studies
    )
    SELECT substr(s.name, 1, length(s.name)-9) AS dataset,
        json_extract(p, '$.embedding_model') AS embedding_model,
        json_extract(p, '$.queried_side') AS side,
        CAST(json_extract(p, '$.top_k') AS INTEGER) AS k,
        json_extract(a, '$.candset_size') AS candidates,
        json_extract(a, '$.tp') AS gold_in_candidates,
        json_extract(a, '$.n_gold') AS gold_pairs,
        1.0*json_extract(a, '$.tp')/json_extract(a, '$.n_gold') AS PC,
        1.0*json_extract(a, '$.tp')/json_extract(a, '$.candset_size') AS PQ
    FROM named s JOIN trials t USING (study_id)
    JOIN params USING (trial_id) JOIN attrs USING (trial_id)
    WHERE s.name GLOB '*_blocking' AND t.state='COMPLETE';

    CREATE TEMP VIEW blocker_operating_evidence AS
    WITH eligible AS (
        SELECT *, row_number() OVER (PARTITION BY dataset, embedding_model
            ORDER BY candidates, PQ DESC, side, k) AS choice
        FROM blocking_evidence WHERE PC >= .90
    ), models AS (
        SELECT DISTINCT dataset, embedding_model FROM blocking_evidence
    )
    SELECT m.*, e.side, e.k, e.candidates, e.gold_in_candidates, e.gold_pairs, e.PC, e.PQ,
        rank() OVER (PARTITION BY m.dataset ORDER BY e.candidates IS NULL, e.candidates) AS rank
    FROM models m LEFT JOIN eligible e ON m.dataset=e.dataset
        AND m.embedding_model=e.embedding_model AND e.choice=1;

    CREATE TEMP VIEW blocker_ranking_scope AS
    SELECT dataset, count(*) AS models, sum(rank=1) AS tied_first,
        2*sum(rank=1)<=count(*) AS used_for_ranking
    FROM blocker_operating_evidence GROUP BY dataset;
    """)
    blocker_ranking = pd.read_sql_query("""
        SELECT o.embedding_model,
            round(avg(CASE WHEN s.used_for_ranking THEN o.rank END),4) AS mean_rank,
            max(CASE WHEN s.used_for_ranking THEN o.rank END) AS worst_rank,
            sum(s.used_for_ranking AND o.rank=1) AS wins,
            round(avg(o.PQ),4) AS mean_PQ,
            sum(o.candidates IS NULL) AS misses
        FROM blocker_operating_evidence o JOIN blocker_ranking_scope s USING (dataset)
        GROUP BY o.embedding_model
        ORDER BY mean_rank IS NULL, mean_rank, worst_rank, mean_PQ DESC, o.embedding_model
    """, blocking_evidence_db)
    blocker_scope = pd.read_sql_query("SELECT * FROM blocker_ranking_scope ORDER BY dataset", blocking_evidence_db)
    blocker_primary_points = pd.read_sql_query("""
        SELECT dataset, side, k, candidates, gold_in_candidates, gold_pairs, PC, PQ
        FROM blocker_operating_evidence WHERE embedding_model=? ORDER BY dataset
    """, blocking_evidence_db, params=(BLOCKER_MODEL,))
    blocker_ladder_points = pd.read_sql_query("""
        SELECT dataset, side, k, candidates, gold_in_candidates, gold_pairs, PC, PQ
        FROM blocking_evidence WHERE embedding_model=?
    """, blocking_evidence_db, params=(BLOCKER_MODEL,))

ladder_cells = {(dataset, side, k) for dataset, cells in LADDER.items() for side, k in cells}
blocker_primary_points["in_existing_ladder"] = [
    (row.dataset, row.side, row.k) in ladder_cells for row in blocker_primary_points.itertuples()]
blocker_ladder_points = blocker_ladder_points.loc[[
    (row.dataset, row.side, row.k) in ladder_cells for row in blocker_ladder_points.itertuples()]]
blocker_ranking["chosen_blocker"] = blocker_ranking.embedding_model.eq(BLOCKER_MODEL)
display(blocker_ranking.head(5))
display(blocker_scope)
display(blocker_primary_points)


,embedding_model,mean_rank,worst_rank,wins,mean_PQ,misses,chosen_blocker
0,tencent_KaLM-Embedding-Gemma3-12B-2511,1.6667,2,1,0.4608,0,True
1,codefuse-ai_F2LLM-v2-14B,1.6667,2,1,0.4595,0,False
2,codefuse-ai_F2LLM-v2-8B,1.6667,2,1,0.4584,0,False
3,codefuse-ai_F2LLM-v2-4B,2.6667,5,1,0.4714,0,False
4,Qwen_Qwen3-Embedding-4B,3.0000,5,0,0.4572,0,False


,dataset,models,tied_first,used_for_ranking
0,abt_buy,20,4,1
1,amazon_googleproducts,20,1,1
2,dblp_acm,20,20,0
3,dblp_scholar,20,1,1
4,fodors_zagats,20,18,0


,dataset,side,k,candidates,gold_in_candidates,gold_pairs,PC,PQ,in_existing_ladder
0,abt_buy,left,2,2162,1012,1097,0.922516,0.468085,True
1,amazon_googleproducts,left,4,5452,1194,1300,0.918462,0.219002,False
2,dblp_acm,right,1,2294,2208,2224,0.992806,0.962511,False
3,dblp_scholar,left,6,15696,4960,5347,0.927623,0.316004,False
4,fodors_zagats,right,1,331,112,112,1.000000,0.338369,True


Magellan baselines reuse saved feature rows by pair ID. The added AG left/k4 cell reuses 4,584 rows and extracts only 868 missing pairs with `prepare.cached_reference_features` under the original Python 3.6 environment. The cache sidecar records the sources and their hashes.

In [7]:
from config import FEATURES_DIR
reuse_path = Path(FEATURES_DIR) / "amazon_googleproducts__left_k4.csv.gz.reuse.json"
if reuse_path.exists():
    feature_reuse = json.loads(reuse_path.read_text())
    display(pd.DataFrame([{key: feature_reuse[key] for key in
        ("rows", "reused_rows", "extracted_rows", "n_features", "extraction_seconds")}]))

,rows,reused_rows,extracted_rows,n_features,extraction_seconds
0,5452,4584,868,48,434.827119
